# 🏥 Clinic Revenue & Patient Visit Analysis
**Author:** Umul Hasanah  
**Tools:** Python, Pandas, Matplotlib  
**Dataset:** Simulated clinic operational data (2023)

---

## 📦 STEP 1 — Load & Cek Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Load dataset
df = pd.read_csv('clinic_data_simulated.csv')

# Lihat 5 baris pertama
print('=== 5 BARIS PERTAMA ===')
print(df.head())

# Cek apakah ada data kosong
print('\n=== CEK DATA KOSONG ===')
print(df.isnull().sum())

# Cek ukuran dataset
print(f'\nTotal baris: {len(df)}, Total kolom: {len(df.columns)}')

## 🧮 STEP 2 — Hitung Angka Penting

In [ ]:
# Ubah revenue ke satuan juta (lebih mudah dibaca)
df['revenue_million'] = df['daily_revenue_idr'] / 1_000_000

# Hitung total biaya operasional
df['total_cost'] = df['petty_cash_used'] + df['supplies_cost']

# Hitung cost ratio (biaya vs pendapatan dalam %)
df['cost_ratio_%'] = (df['total_cost'] / df['daily_revenue_idr']) * 100

# Hitung insurance claim approval rate
df['claim_approval_%'] = (df['insurance_claims_paid'] / df['insurance_claims_submitted']) * 100

# Tampilkan hasil
print('=== RINGKASAN KEUANGAN PER BULAN ===')
print(df[['month','revenue_million','total_cost','cost_ratio_%']].round(2).to_string(index=False))

In [ ]:
# Cari bulan dengan cost ratio tinggi (di atas 20%)
high_cost = df[df['cost_ratio_%'] > 20]

print('=== BULAN DENGAN BIAYA TINGGI (cost ratio > 20%) ===')
if len(high_cost) > 0:
    print(high_cost[['month','revenue_million','cost_ratio_%']].round(2).to_string(index=False))
else:
    print('Tidak ada bulan dengan cost ratio di atas 20%')

# Statistik ringkas
print(f'\n=== STATISTIK TAHUNAN ===')
print(f'Total Revenue 2023     : IDR {df["daily_revenue_idr"].sum()/1_000_000:.1f} Juta')
print(f'Rata-rata Revenue/bulan: IDR {df["daily_revenue_idr"].mean()/1_000_000:.1f} Juta')
print(f'Bulan Revenue Tertinggi: {df.loc[df["revenue_million"].idxmax(), "month"]}')
print(f'Bulan Revenue Terendah : {df.loc[df["revenue_million"].idxmin(), "month"]}')
print(f'Rata-rata Cost Ratio   : {df["cost_ratio_%"].mean():.1f}%')
print(f'Rata-rata Claim Approval: {df["claim_approval_%"].mean():.1f}%')

## 📈 STEP 3 — Visualisasi Grafik

In [ ]:
# ── GRAFIK 1: Revenue Trend ──────────────────────────
plt.figure(figsize=(12, 5))
plt.plot(df['month'], df['revenue_million'],
         marker='o', color='#1F4E79', linewidth=2.5, markersize=7)
plt.fill_between(range(len(df)), df['revenue_million'],
                 alpha=0.1, color='#1F4E79')

# Tambahkan label angka di setiap titik
for i, row in df.iterrows():
    plt.annotate(f"{row['revenue_million']:.1f}M",
                 (i, row['revenue_million']),
                 textcoords='offset points', xytext=(0,10),
                 ha='center', fontsize=8, color='#1F4E79')

plt.title('Monthly Clinic Revenue 2023 (IDR Million)', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Revenue (IDR Million)')
plt.xticks(range(len(df)), df['month'], rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('revenue_trend.png', dpi=150)
plt.show()
print('✅ Grafik revenue_trend.png tersimpan!')

In [ ]:
# ── GRAFIK 2: Patient Visits (Stacked Bar) ───────────
fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(df['month'], df['general_patients'],
       label='General Patients', color='#1F4E79')
ax.bar(df['month'], df['insurance_patients'],
       bottom=df['general_patients'],
       label='Insurance Patients', color='#5BA3D0')

ax.set_title('Monthly Patient Visits by Type 2023', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Patients')
ax.legend()
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('patient_visits.png', dpi=150)
plt.show()
print('✅ Grafik patient_visits.png tersimpan!')

In [ ]:
# ── GRAFIK 3: Cost Ratio per Bulan ───────────────────
colors = ['#E74C3C' if x > 20 else '#1F4E79' for x in df['cost_ratio_%']]

plt.figure(figsize=(12, 5))
bars = plt.bar(df['month'], df['cost_ratio_%'], color=colors)
plt.axhline(y=20, color='red', linestyle='--', alpha=0.7, label='Threshold 20%')

# Label angka di atas bar
for bar, val in zip(bars, df['cost_ratio_%']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=8)

plt.title('Monthly Cost-to-Revenue Ratio 2023 (%)', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Cost Ratio (%)')
plt.legend()
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('cost_ratio.png', dpi=150)
plt.show()
print('✅ Grafik cost_ratio.png tersimpan!')

## 💡 STEP 4 — Insight & Kesimpulan Bisnis

In [ ]:
print('=' * 55)
print('       KEY INSIGHTS & BUSINESS RECOMMENDATIONS')
print('=' * 55)

# Insight 1
low_month = df.loc[df['revenue_million'].idxmin(), 'month']
low_rev = df['revenue_million'].min()
high_cost_ratio = df.loc[df['revenue_million'].idxmin(), 'cost_ratio_%']
print(f"""
📌 INSIGHT 1 — Low Season Problem
   {low_month} had the lowest revenue (IDR {low_rev:.1f}M)
   but the highest cost ratio ({high_cost_ratio:.1f}%).
   → Recommendation: Increase patient acquisition
     campaigns in February (promo via WhatsApp/IG).
""")

# Insight 2
peak_month = df.loc[df['revenue_million'].idxmax(), 'month']
peak_rev = df['revenue_million'].max()
print(f"""📌 INSIGHT 2 — Peak Season
   {peak_month} recorded the highest revenue (IDR {peak_rev:.1f}M).
   → Recommendation: Prepare adequate staff and
     supplies in advance for peak months.
""")

# Insight 3
avg_approval = df['claim_approval_%'].mean()
print(f"""📌 INSIGHT 3 — Insurance Claim Performance
   Average claim approval rate: {avg_approval:.1f}%.
   → This indicates strong documentation compliance.
     Maintain SOP checklist to sustain this rate.
""")

print('=' * 55)

---
## ✅ Project Complete!

File yang dihasilkan:
- `revenue_trend.png` — grafik tren pendapatan
- `patient_visits.png` — grafik kunjungan pasien
- `cost_ratio.png` — grafik rasio biaya

Upload semua file ini ke GitHub repository kamu! 🚀